# Federated Learning DP vs No-DP Experiment Analysis

This notebook reads the terminal output text files from the federated learning experiments and extracts the key numbers needed for the final report.

It compares four experiments:

1. **No-DP**: no client noise and no server noise  
2. **Local DP Only**: client noise only  
3. **Global DP Only**: server noise only  
4. **Both DP**: client noise + server noise  

The main metrics extracted are:

- **Average local accuracy before server aggregation**
- **Demographic Parity Gap** for age-group fairness
- Final-round comparison across all experiments

This is important for the final report because the project is about the trade-off between **privacy, accuracy, and fairness** in federated learning.


In [1]:
# Cell 1: Import required libraries

# pandas is used for tables and CSV outputs.
# matplotlib is used for report-ready graphs.
# re is used to extract numbers from terminal output text.

import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


## Step 1: Define the experiment output files

Put this notebook in the same folder as these text files:

- `non_dp.txt`
- `local_only.txt`
- `global_only.txt`
- `dp.txt`

These files should contain the copied terminal output from each experiment run.


In [2]:
# Cell 2: Define experiment files and DP settings

log_files = {
    "No-DP": {
        "path": "non_dp.txt",
        "client_noise": 0.0,
        "server_noise": 0.0
    },
    "Local DP Only": {
        "path": "local_only.txt",
        "client_noise": 0.1,
        "server_noise": 0.0
    },
    "Global DP Only": {
        "path": "global_only.txt",
        "client_noise": 0.0,
        "server_noise": 0.05
    },
    "Both DP": {
        "path": "dp.txt",
        "client_noise": 0.1,
        "server_noise": 0.05
    }
}

for name, info in log_files.items():
    print(f"{name}: {info['path']}")


No-DP: non_dp.txt
Local DP Only: local_only.txt
Global DP Only: global_only.txt
Both DP: dp.txt


## Step 2: Parse one experiment log

This function extracts the important values from a single terminal output file.

It extracts:

- round number
- average local accuracy
- all demographic parity gaps printed in that round
- average fairness gap for the round

Since gender and race metrics often showed `None` in your outputs, this notebook uses the available numeric **Demographic Parity Gap** values, which mainly come from age-group fairness.


In [3]:
# Cell 3: Function to parse experiment output text files

def parse_experiment_log(file_path):
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"Could not find file: {file_path.resolve()}")

    text = file_path.read_text(errors="ignore")

    # Extract round numbers
    round_headers = re.findall(r"ROUND\s+(\d+)", text)

    # Split text into blocks by round
    round_blocks = re.split(r"=+\s*\\nROUND\s+\\d+\\s*\\n=+", text)

    # The first block is before ROUND 1, so ignore it
    round_blocks = round_blocks[1:]

    rows = []

    for idx, block in enumerate(round_blocks):
        round_num = int(round_headers[idx]) if idx < len(round_headers) else idx + 1

        # Extract average local accuracy
        acc_match = re.search(
            r"Average local accuracy before server aggregation:\\s*([0-9.]+)",
            block
        )
        avg_accuracy = float(acc_match.group(1)) if acc_match else None

        # Extract client overall accuracies
        client_accs = re.findall(
            r"Client\\s+(\\d+)\\s+Overall Accuracy:\\s*([0-9.]+)",
            block
        )

        client_acc_dict = {
            f"client_{client_id}_accuracy": float(acc)
            for client_id, acc in client_accs
        }

        # Extract all numeric demographic parity gaps
        dp_gaps = re.findall(r"Demographic Parity Gap:\\s*([0-9.]+)", block)
        dp_gaps = [float(gap) for gap in dp_gaps]

        avg_dp_gap = sum(dp_gaps) / len(dp_gaps) if dp_gaps else None
        min_dp_gap = min(dp_gaps) if dp_gaps else None
        max_dp_gap = max(dp_gaps) if dp_gaps else None

        row = {
            "round": round_num,
            "avg_accuracy": avg_accuracy,
            "avg_demographic_parity_gap": avg_dp_gap,
            "min_demographic_parity_gap": min_dp_gap,
            "max_demographic_parity_gap": max_dp_gap,
            "num_fairness_gaps_found": len(dp_gaps),
        }

        row.update(client_acc_dict)
        rows.append(row)

    return pd.DataFrame(rows)


## Step 3: Parse all four experiments

This cell reads all four `.txt` files and combines them into one dataframe.

This is important because the final report needs a clean comparison across DP settings, not separate terminal screenshots.


In [4]:
# Cell 4: Parse all experiments

all_results = []
missing_files = []

for experiment_name, info in log_files.items():
    try:
        df = parse_experiment_log(info["path"])
        df["experiment"] = experiment_name
        df["client_noise"] = info["client_noise"]
        df["server_noise"] = info["server_noise"]
        all_results.append(df)
    except FileNotFoundError as e:
        missing_files.append(str(e))

if missing_files:
    print("Missing files:")
    for error in missing_files:
        print(error)

if not all_results:
    raise ValueError("No experiment files were found. Check that the text files are in the same folder as this notebook.")

results_df = pd.concat(all_results, ignore_index=True)

# Reorder important columns first
important_cols = [
    "experiment",
    "client_noise",
    "server_noise",
    "round",
    "avg_accuracy",
    "avg_demographic_parity_gap",
    "min_demographic_parity_gap",
    "max_demographic_parity_gap",
    "num_fairness_gaps_found",
]

other_cols = [c for c in results_df.columns if c not in important_cols]
results_df = results_df[important_cols + other_cols]

results_df


KeyError: "['round', 'avg_accuracy', 'avg_demographic_parity_gap', 'min_demographic_parity_gap', 'max_demographic_parity_gap', 'num_fairness_gaps_found'] not in index"

## Step 4: Create the final-round comparison table

For the final report, the most important row is usually the last communication round because it represents the final trained model state.

This table should go directly into the Results section of the report.


In [ ]:
# Cell 5: Final-round comparison table

final_round_df = (
    results_df
    .sort_values(["experiment", "round"])
    .groupby("experiment", as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

final_round_df = final_round_df[
    [
        "experiment",
        "client_noise",
        "server_noise",
        "round",
        "avg_accuracy",
        "avg_demographic_parity_gap",
        "min_demographic_parity_gap",
        "max_demographic_parity_gap",
        "num_fairness_gaps_found",
    ]
]

final_round_df


## Step 5: Save clean result tables

These CSV files can be submitted with the report or used to make tables in Word/Google Docs.

- `all_experiment_results.csv`: all rounds from all experiments
- `final_round_comparison.csv`: final-round summary only


In [ ]:
# Cell 6: Save CSV outputs

results_df.to_csv("all_experiment_results.csv", index=False)
final_round_df.to_csv("final_round_comparison.csv", index=False)

print("Saved: all_experiment_results.csv")
print("Saved: final_round_comparison.csv")


## Step 6: Plot accuracy across rounds

This graph shows how model accuracy changes across FL rounds for each DP setting.

Use this in the final report to discuss the **accuracy cost of differential privacy**.


In [ ]:
# Cell 7: Accuracy across rounds

plt.figure(figsize=(8, 5))

for experiment in results_df["experiment"].unique():
    temp = results_df[results_df["experiment"] == experiment]
    plt.plot(temp["round"], temp["avg_accuracy"], marker="o", label=experiment)

plt.xlabel("Federated Learning Round")
plt.ylabel("Average Local Accuracy")
plt.title("Accuracy Across Federated Learning Rounds")
plt.ylim(0, 1)
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig("accuracy_across_rounds.png", dpi=300)
plt.show()


## Step 7: Plot fairness gap across rounds

This graph shows how Demographic Parity Gap changes over FL rounds.

Lower values mean better fairness. This graph helps explain whether DP noise improved or worsened fairness.


In [ ]:
# Cell 8: Fairness gap across rounds

plt.figure(figsize=(8, 5))

for experiment in results_df["experiment"].unique():
    temp = results_df[results_df["experiment"] == experiment]
    plt.plot(
        temp["round"],
        temp["avg_demographic_parity_gap"],
        marker="o",
        label=experiment
    )

plt.xlabel("Federated Learning Round")
plt.ylabel("Average Demographic Parity Gap")
plt.title("Fairness Gap Across Federated Learning Rounds")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig("fairness_gap_across_rounds.png", dpi=300)
plt.show()


## Step 8: Final accuracy comparison

This bar chart compares final accuracy across No-DP, Local DP, Global DP, and Both DP.

Use this graph in the final report to show whether privacy noise reduced model utility.


In [ ]:
# Cell 9: Final accuracy comparison

plt.figure(figsize=(7, 5))

plt.bar(final_round_df["experiment"], final_round_df["avg_accuracy"])

plt.xlabel("Experiment")
plt.ylabel("Final Average Accuracy")
plt.title("Final Accuracy Comparison")
plt.ylim(0, 1)
plt.grid(axis="y")
plt.tight_layout()

plt.savefig("final_accuracy_comparison.png", dpi=300)
plt.show()


## Step 9: Final fairness gap comparison

This bar chart compares the final fairness gap across experiments.

Use this graph to discuss the **fairness impact of DP**.


In [ ]:
# Cell 10: Final fairness gap comparison

plt.figure(figsize=(7, 5))

plt.bar(
    final_round_df["experiment"],
    final_round_df["avg_demographic_parity_gap"]
)

plt.xlabel("Experiment")
plt.ylabel("Final Average Demographic Parity Gap")
plt.title("Final Fairness Gap Comparison")
plt.grid(axis="y")
plt.tight_layout()

plt.savefig("final_fairness_gap_comparison.png", dpi=300)
plt.show()


## Step 10: Report-ready written summary

This cell prints a short summary that can be copied into the report draft.

You should still explain the meaning in your own words, but this gives the key numbers.


In [ ]:
# Cell 11: Report-ready summary text

for _, row in final_round_df.iterrows():
    print(
        f"{row['experiment']}: "
        f"client noise={row['client_noise']}, "
        f"server noise={row['server_noise']}, "
        f"final round={int(row['round'])}, "
        f"final avg accuracy={row['avg_accuracy']:.4f}, "
        f"final avg demographic parity gap={row['avg_demographic_parity_gap']:.4f}"
    )


## Files generated by this notebook

After running the notebook, these files are created:

### Tables
- `all_experiment_results.csv`
- `final_round_comparison.csv`

### Report-ready figures
- `accuracy_across_rounds.png`
- `fairness_gap_across_rounds.png`
- `final_accuracy_comparison.png`
- `final_fairness_gap_comparison.png`

Use the final-round CSV table and the four images in your final report.
